# 세그멘테이션 수치 검증 — 검정 · 분류 · 결정 트리 · 연관분석 · 기대 효과

설계: [`reports/MODELING_PLAN.md`](../reports/MODELING_PLAN.md) 1~5단계.
입력: `data/processed/InsightStay_data_cleaned.parquet` (`scripts/build_master.py` 실행 결과 — `segment`, `L3_new`, `price_rel`, `amenity_rel`, `quote_peak` 컬럼이 있어야 합니다).

| 단계 | 질문 | 기법 |
|---|---|---|
| 1 | 층 사이 차이가 우연인가? | Kruskal-Wallis · 카이제곱 + 효과 크기 |
| 2 | 무엇이 L3(단기 무거래)를 만드나? | 로지스틱 회귀 (오즈비 · 95% 신뢰구간 · 도시 단위 검증) |
| 3 | 누구에게 개입하나? | 결정 트리 (분기점 · 안정성) |
| 4 | 무엇을 바꾸라고 하나? | 연관분석 (지지도 · 향상도 · 동급 내 거래율 차이) |
| 5 | 바꾸면 얼마를 버나? | 회귀 + 시나리오 (반응률 · 리뷰 작성률 범위) |

In [ ]:
# [0-1] 라이브러리 & 기본 설정 — eda.ipynb 와 같은 스타일
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

for _f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]:
    if any(_f == f.name for f in mpl.font_manager.fontManager.ttflist):
        mpl.rcParams["font.family"] = _f
        break
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["figure.dpi"] = 110
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.alpha"] = 0.25
mpl.rcParams["axes.spines.top"] = False
mpl.rcParams["axes.spines.right"] = False
C = {"main": "#4E79A7", "accent": "#F28E2B", "ok": "#59A14F", "bad": "#E15759", "gray": "#9C9C9C"}

ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SEED = 0

In [ ]:
# [0-2] 데이터 로드 — 필요한 컬럼만
cols = ["id", "source", "region", "room_type", "segment", "L3_new", "minimum_nights",
        "price", "price_rel", "amenity_count", "amenity_rel", "quote_peak",
        "host_is_superhost", "number_of_reviews_ltm"]
df = pd.read_parquet(ROOT / "data/processed/InsightStay_data_cleaned.parquet", columns=cols)
missing = {"segment", "L3_new", "price_rel", "amenity_rel", "quote_peak"} - set(df.columns)
assert not missing, f"build_master.py 를 다시 실행하세요: {missing}"
print(f"{len(df):,}행")
df["segment"].value_counts().sort_index()

## 1. 통계 검정 — 층 차이가 우연이 아니다

- 대상: **L2~L5, `source == 0`**. L1은 99.8%가 이월값이라 특성 비교에서 뺍니다([SEGMENTATION.md](../reports/SEGMENTATION.md) L1 카드).
- 표본이 수십만 건이라 **p값은 무조건 작게 나옵니다.** 그래서 **효과 크기**를 같이 봅니다.
  - Kruskal-Wallis → ε² (0.01 작음 · 0.06 중간 · 0.14 큼)
  - 카이제곱 → Cramér's V (0.1 작음 · 0.3 중간 · 0.5 큼)

In [ ]:
# [1-1] 층별 요약
t = df[(df["source"] == 0) & df["segment"].isin(["L2", "L3", "L4", "L5"])].copy()
sh = t["host_is_superhost"].where(t["host_is_superhost"] != "unknown")   # unknown 은 분모에서 뺀다
t["superhost"] = sh.map({"t": 1, "f": 0})
t["entire"] = (t["room_type"] == "Entire home/apt").astype(int)

summary = t.groupby("segment").agg(
    n=("id", "size"),
    편의시설_중앙=("amenity_count", "median"),
    가격_중앙=("price", "median"),
    슈퍼호스트=("superhost", "mean"),
    통집=("entire", "mean"),
)
summary

In [ ]:
# [1-2] 검정 + 효과 크기
def kruskal_eps2(frame, col):
    groups = [g[col].dropna().to_numpy() for _, g in frame.groupby("segment")]
    h, p = stats.kruskal(*groups)
    n, k = sum(len(g) for g in groups), len(groups)
    return h, p, (h - k + 1) / (n - k), n

def chi2_cramer(frame, col):
    ct = pd.crosstab(frame["segment"], frame[col].dropna())
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    v = np.sqrt(chi2 / (ct.to_numpy().sum() * (min(ct.shape) - 1)))
    return chi2, p, v, int(ct.to_numpy().sum())

rows = []
for col, name in [("amenity_count", "편의시설 개수"), ("price", "1박 가격")]:
    h, p, e2, n = kruskal_eps2(t, col)
    rows.append({"변수": name, "검정": "Kruskal-Wallis", "통계량": h, "p": p, "효과크기": e2, "지표": "ε²", "n": n})
for col, name in [("superhost", "슈퍼호스트"), ("entire", "통집")]:
    c2, p, v, n = chi2_cramer(t, col)
    rows.append({"변수": name, "검정": "카이제곱", "통계량": c2, "p": p, "효과크기": v, "지표": "Cramér V", "n": n})
tests = pd.DataFrame(rows).set_index("변수")
tests

In [ ]:
# [1-3] 핵심 비교 L3 vs L5 — 발표에서 쓰는 쌍
l3, l5 = t[t["segment"] == "L3"], t[t["segment"] == "L5"]
pair = []
for col, name in [("amenity_count", "편의시설 개수"), ("price", "1박 가격")]:
    a, b = l3[col].dropna(), l5[col].dropna()
    u, p = stats.mannwhitneyu(a, b)
    # 순위이연상관: L3 값이 L5 값보다 클 확률 − 작을 확률
    rbc = 2 * u / (len(a) * len(b)) - 1
    pair.append({"변수": name, "L3 중앙값": a.median(), "L5 중앙값": b.median(),
                 "순위이연상관": rbc, "p": p})
pd.DataFrame(pair).set_index("변수")

**읽는 법** — 순위이연상관이 음수면 L3가 L5보다 작은 쪽, 양수면 큰 쪽입니다(−1~1).
L3는 **편의시설은 L5보다 적고 가격은 높습니다.** 두 방향이 반대라는 것이 L3의 모순입니다.

## 2. 로지스틱 회귀 — 무엇이 L3를 만드나

| 항목 | 설계 |
|---|---|
| 대상 | L3 + L4 + L5 · `source == 0` · 최소 숙박일 30박 미만 · `price_rel`·`amenity_rel`·`quote_peak` 보유 · **`L3_new` 제외** |
| 목표 | L3인가 (최근 1년 무거래) |
| 설명 변수 | 동급 대비 가격(로그) · 동급 대비 편의시설(5개 단위) · 최소 숙박일 구간(기준 2박) · 개인실 · 성수기 견적 |
| 뺀 변수 | 층 정의에 쓴 세 컬럼(순환 논리) · 슈퍼호스트(거래의 결과일 수 있음) · 평점(무거래 매물은 절반이 결측) |

최소 숙박일 30박 이상을 빼는 이유: L3는 정의상 30박 미만이라, 넣으면 "30박 이상 = L3 아님"을 모델이 공짜로 맞힙니다.

In [ ]:
# [2-1] 모델링 표본
m = df[(df["source"] == 0) & df["segment"].isin(["L3", "L4", "L5"]) & (df["minimum_nights"] < 30)
       & df["price_rel"].notna() & df["amenity_rel"].notna() & df["quote_peak"].notna()
       & ~df["L3_new"]].copy()
m["y"] = (m["segment"] == "L3").astype(int)
mn = pd.cut(m["minimum_nights"], [0, 1, 2, 3, 7, 29], labels=["1", "2", "3", "4-7", "8-29"])

X = pd.DataFrame({
    "동급 대비 가격 (log)":       np.log(m["price_rel"].clip(0.2, 5)),
    "동급 대비 편의시설 (+5개)":   m["amenity_rel"].clip(-40, 40) / 5,
    "최소 숙박 1박":             (mn == "1").astype(int),
    "최소 숙박 3박":             (mn == "3").astype(int),
    "최소 숙박 4-7박":           (mn == "4-7").astype(int),
    "최소 숙박 8-29박":          (mn == "8-29").astype(int),
    "개인실·호텔·공유":           (m["room_type"] != "Entire home/apt").astype(int),
    "성수기(6-8월) 견적":         m["quote_peak"].astype(int),
}, index=m.index)
y = m["y"]
print(f"표본 {len(m):,}건 · L3 비율 {y.mean():.1%} · 도시 {m['region'].nunique()}개")

In [ ]:
# [2-2] 적합 + 오즈비 95% 신뢰구간
# 규제 없는 로지스틱을 쓰고, 표준오차는 피셔 정보행렬 (X'WX)^-1 에서 직접 구한다.
def fit_logit(X, y):
    model = LogisticRegression(penalty=None, max_iter=5000).fit(X, y)
    p = model.predict_proba(X)[:, 1]
    Xd = np.column_stack([np.ones(len(X)), X.to_numpy(dtype=float)])
    cov = np.linalg.inv(Xd.T @ (Xd * (p * (1 - p))[:, None]))
    se = np.sqrt(np.diag(cov))[1:]
    return model, se

logit, se = fit_logit(X, y)
b = logit.coef_[0]
coef = pd.DataFrame({
    "계수": b, "표준오차": se, "z": b / se,
    "오즈비": np.exp(b), "95% 하한": np.exp(b - 1.96 * se), "95% 상한": np.exp(b + 1.96 * se),
}, index=X.columns)

# 표준화 계수 — 단위가 다른 변수끼리 크기를 비교하려고 각 변수를 표준편차 1로 맞춘 뒤 다시 적합
Z = (X - X.mean()) / X.std()
coef["표준화 계수"] = LogisticRegression(penalty=None, max_iter=5000).fit(Z, y).coef_[0]
coef.sort_values("표준화 계수", key=abs, ascending=False)

In [ ]:
# [2-3] 검증 — ① 무작위 7:3  ② 도시 단위 5겹 (학습에 쓰지 않은 도시에서 평가)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
auc_random = roc_auc_score(yte, LogisticRegression(penalty=None, max_iter=5000)
                           .fit(Xtr, ytr).predict_proba(Xte)[:, 1])

city_auc = []
for tr, te in GroupKFold(n_splits=5).split(X, y, groups=m["region"]):
    mdl = LogisticRegression(penalty=None, max_iter=5000).fit(X.iloc[tr], y.iloc[tr])
    city_auc.append(roc_auc_score(y.iloc[te], mdl.predict_proba(X.iloc[te])[:, 1]))

# 변수 하나만 넣었을 때의 AUC — 어느 쪽이 혼자서 더 잘 가르나
solo = {c: roc_auc_score(yte, LogisticRegression().fit(Xtr[[c]], ytr).predict_proba(Xte[[c]])[:, 1])
        for c in ["동급 대비 가격 (log)", "동급 대비 편의시설 (+5개)"]}

print(f"무작위 분할 AUC       {auc_random:.3f}")
print(f"도시 단위 5겹 AUC     {np.mean(city_auc):.3f}  (겹별 {', '.join(f'{a:.3f}' for a in city_auc)})")
for k, v in solo.items():
    print(f"단독 AUC · {k:<18} {v:.3f}")

In [ ]:
# [2-4] 오즈비 그림 — 1보다 크면 L3가 될 가능성↑, 작으면↓ (로그 축)
order = coef.sort_values("오즈비").index
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ypos = np.arange(len(order))
ax.errorbar(coef.loc[order, "오즈비"], ypos,
            xerr=[coef.loc[order, "오즈비"] - coef.loc[order, "95% 하한"],
                  coef.loc[order, "95% 상한"] - coef.loc[order, "오즈비"]],
            fmt="o", ms=8, color=C["main"], ecolor=C["main"], elinewidth=2, capsize=0)
ax.axvline(1, color=C["gray"], lw=1)
ax.set_xscale("log")
ticks = [0.5, 0.7, 1, 1.5, 2, 3, 4]
ax.set_xticks(ticks, [f"{t:g}" for t in ticks])
ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
ax.set_yticks(ypos, order)
ax.set_xlabel("오즈비 (L3가 될 가능성, 로그 축)")
ax.set_title("L3(단기 무거래)가 될 가능성을 바꾸는 요인", loc="left")
for yi, name in zip(ypos, order):
    ax.annotate(f"{coef.loc[name, '오즈비']:.2f}", (coef.loc[name, "95% 상한"], yi),
                xytext=(6, 0), textcoords="offset points", va="center", fontsize=9, color="#444")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

**읽는 법**
- **표준화 계수**는 "변수를 표준편차 하나만큼 움직였을 때의 효과"라 변수끼리 크기를 비교할 수 있습니다. 편의시설과 가격 중 어느 쪽 절댓값이 큰지가 발표 메시지입니다.
- **오즈비**는 해석용입니다. 예: 편의시설 오즈비 0.8 = 동급보다 5개 많을 때마다 L3가 될 오즈가 20% 줄어듦.
- 가격은 로그로 넣었으므로 오즈비는 "로그 1단위(= 2.72배)"당 값입니다. **동급보다 2배 비쌀 때의 오즈비는 `2 ** 계수`** 로 계산합니다.
- 성수기 견적의 오즈비가 1보다 작은 것은, 비수기 날짜로 견적이 잡힌 매물(성수기 예약이 이미 찬 매물이 아님)에 무거래가 더 많다는 뜻입니다. 이 변수는 계절 효과를 **통제하려고** 넣은 것이라 해석의 중심에 두지 않습니다.
- 이 모델은 **인과가 아니라 연관**을 보여줍니다. AUC가 0.7 안팎이면 예측 도구로는 보통이라 "요인 크기를 나누어 보는 도구"로 소개합니다.

### 2-5. 견고성 — 편의시설 개수에 '목록 갱신 여부'가 섞여 있다

플랫폼이 편의시설 이름 체계를 바꿨습니다. 옛 이름(`Washer`, `TV`, `Air conditioning`)과 새 이름(`Free washer – In unit`, `TV with standard cable`, `AC - split type ductless system`)이 있고, **`Washer`와 `Free washer`를 동시에 가진 매물은 0건**입니다.
새 체계는 항목이 더 잘게 나뉘어 있어서, **목록을 최근에 고친 매물일수록 편의시설 개수가 많게 잡힙니다.** 그래서 이름 체계를 통제해도 편의시설 효과가 남는지 확인합니다.

In [ ]:
# [2-5] 편의시설 이름 체계 — 옛 이름만 / 새 이름만 / 둘 다
import json
OLD_NAMES = {"Washer", "Dryer", "TV", "Air conditioning", "Patio or balcony"}
NEW_PREFIX = ("Free washer", "Paid washer", "Free dryer", "Paid dryer", "AC - ", "Central air conditioning",
              "Private patio or balcony", "Shared patio or balcony", "TV with", "HDTV")
am = pd.read_parquet(ROOT / "data/processed/InsightStay_data_cleaned.parquet",
                     columns=["amenities"]).loc[m.index, "amenities"].map(json.loads)
m["schema_old"] = am.map(lambda a: any(x in OLD_NAMES for x in a))
m["schema_new"] = am.map(lambda a: any(x.startswith(NEW_PREFIX) for x in a))
schema = np.select([m["schema_old"] & ~m["schema_new"], m["schema_new"] & ~m["schema_old"],
                    m["schema_old"] & m["schema_new"]], ["옛 이름만", "새 이름만", "둘 다"], "판별 불가")
print(pd.DataFrame({"schema": schema, "y": y, "amen": m["amenity_count"]})
      .groupby("schema").agg(n=("y", "size"), L3비율=("y", "mean"), 편의시설_중앙=("amen", "median")))

# 이름 체계 더미를 넣고 다시 적합 — 편의시설 표준화 계수가 얼마나 줄어드나
X2 = X.assign(**{"새 이름 체계": m["schema_new"].astype(int), "옛 이름 체계": m["schema_old"].astype(int)})
Z2 = (X2 - X2.mean()) / X2.std()
std2 = pd.Series(LogisticRegression(penalty=None, max_iter=5000).fit(Z2, y).coef_[0], index=X2.columns)
pd.DataFrame({"기존": coef["표준화 계수"], "이름 체계 통제": std2}).loc[
    ["동급 대비 편의시설 (+5개)", "동급 대비 가격 (log)", "새 이름 체계", "옛 이름 체계"]]

**읽는 법** — 이름 체계를 통제해도 편의시설 계수가 가격 계수보다 크면 "상품력 > 가격" 결론이 유지됩니다.
다만 줄어든 만큼은 **"목록을 관리하지 않는 호스트"** 효과였다는 뜻입니다. 발표에서는 "상품력·리스팅 관리 상태"로 묶어 말하는 것이 정확합니다.

## 3. 결정 트리 — 누구에게 개입하나

해석할 수 있도록 **변수 3개(동급 대비 가격 · 동급 대비 편의시설 · 최소 숙박일), 깊이 3, 잎 하나당 5,000건 이상**으로 제한합니다.
사람이 기준선을 정하지 않고 **트리가 데이터에서 찾은 분기점**을 알림 대상 선정 규칙으로 씁니다.

In [ ]:
# [3-1] 적합
T = pd.DataFrame({"price_rel": m["price_rel"], "amenity_rel": m["amenity_rel"],
                  "minimum_nights": m["minimum_nights"]})
Ttr, Tte = T.loc[Xtr.index], T.loc[Xte.index]
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5000, random_state=SEED).fit(Ttr, ytr)
print(f"트리 테스트 AUC {roc_auc_score(yte, tree.predict_proba(Tte)[:, 1]):.3f}")
print(export_text(tree, feature_names=list(T.columns), decimals=2))

In [ ]:
# [3-2] 잎별 L3 비율 — 규칙을 사람이 읽는 문장으로
def leaf_rules(tree, names):
    t = tree.tree_
    out = {}
    def walk(node, conds):
        if t.children_left[node] == -1:
            out[node] = " & ".join(conds) or "(전체)"
            return
        f, thr = names[t.feature[node]], t.threshold[node]
        walk(t.children_left[node], conds + [f"{f} ≤ {thr:.2f}"])
        walk(t.children_right[node], conds + [f"{f} > {thr:.2f}"])
    walk(0, [])
    return out

rules = leaf_rules(tree, list(T.columns))
leaf = pd.Series(tree.apply(T), index=T.index)
leaves = (pd.DataFrame({"leaf": leaf, "y": y})
          .groupby("leaf")["y"].agg(n="size", L3비율="mean")
          .assign(규칙=lambda d: d.index.map(rules),
                  전체대비=lambda d: d["L3비율"] / y.mean())
          .sort_values("L3비율", ascending=False))
print(f"전체 L3 비율 {y.mean():.1%}")
leaves

In [ ]:
# [3-3] 분기점 안정성 — 표본을 절반씩 5번 다시 뽑아도 같은 분기점이 나오나
def splits(tree, names):
    t = tree.tree_
    return [(names[t.feature[i]], round(t.threshold[i], 2))
            for i in range(t.node_count) if t.children_left[i] != -1]

stab = []
for s in range(5):
    idx = T.sample(frac=0.5, random_state=100 + s).index
    tr_ = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5000, random_state=SEED).fit(T.loc[idx], y.loc[idx])
    sp = splits(tr_, list(T.columns))
    stab.append({"표본": s + 1,
                 "가격 분기점": sorted({v for f, v in sp if f == "price_rel"}),
                 "편의시설 분기점": sorted({v for f, v in sp if f == "amenity_rel"}),
                 "최소숙박 분기점": sorted({v for f, v in sp if f == "minimum_nights"})})
pd.DataFrame(stab).set_index("표본")

In [ ]:
# [3-4] 트리 그림
fig, ax = plt.subplots(figsize=(15, 6.5))
plot_tree(tree, feature_names=["동급 대비 가격", "동급 대비 편의시설", "최소 숙박일"],
          class_names=["거래", "L3"], filled=False, impurity=False, proportion=True,
          rounded=True, precision=2, fontsize=8, ax=ax)
ax.set_title("L3 판별 결정 트리 (깊이 3) — value = [거래, L3] 비율", loc="left")
plt.tight_layout(); plt.show()

In [ ]:
# [3-5] 트리 분기점으로 본 L3 비율 — 가격 × 편의시설 (최소 숙박일 4박 이하 / 5박 이상 나눠서)
pb = pd.cut(m["price_rel"], [0, .7, .85, 1, 1.15, 1.3, 1.6, 2.4, np.inf],
            labels=["<0.7", "0.7-0.85", "0.85-1", "1-1.15", "1.15-1.3", "1.3-1.6", "1.6-2.4", "2.4+"])
ab = pd.cut(m["amenity_rel"], [-np.inf, -13, -8, -3, 3, 8, np.inf],
            labels=["-13 미만", "-13~-8", "-8~-3", "±3", "+3~+8", "+8 초과"])
long_stay = np.where(m["minimum_nights"] >= 5, "최소 숙박 5박 이상", "최소 숙박 4박 이하")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, key in zip(axes, ["최소 숙박 4박 이하", "최소 숙박 5박 이상"]):
    sel = long_stay == key
    piv = pd.crosstab(ab[sel], pb[sel], values=y[sel], aggfunc="mean")
    im = ax.imshow(piv.to_numpy(), cmap="Blues", vmin=0, vmax=0.8, aspect="auto")
    ax.set_xticks(range(piv.shape[1]), piv.columns, rotation=30, ha="right")
    ax.set_yticks(range(piv.shape[0]), piv.index)
    ax.set_xlabel("동급 대비 가격 (배)")
    ax.set_title(key, loc="left")
    ax.grid(False)
axes[0].set_ylabel("동급 대비 편의시설 (개)")
fig.colorbar(im, ax=axes, label="L3 비율", shrink=0.9)
fig.suptitle("가격이 높고 편의시설이 적을수록, 최소 숙박일이 길수록 L3 비율이 높다", x=0.01, y=1.04, ha="left")
plt.show()

In [ ]:
# [3-6] 위 그림의 표 — 최소 숙박 4박 이하
pd.crosstab(ab[long_stay == "최소 숙박 4박 이하"], pb[long_stay == "최소 숙박 4박 이하"],
            values=y[long_stay == "최소 숙박 4박 이하"], aggfunc="mean").round(2)

## 4. 연관분석 — 무엇을 바꾸라고 하나

규칙 **{편의시설 X} → {거래}** 의 지지도 · 향상도(lift)를 구하고, 도시·방 타입마다 기본 구비가 다르므로 **동급 그룹 안에서 보유/미보유 매물의 거래율 차이**를 함께 봅니다.

- **새 이름 체계를 쓰는 매물만** 씁니다. 옛 체계 매물에는 새 항목이 선택지로 없었을 수 있어서, 섞으면 "항목이 없다"와 "목록이 오래됐다"가 구분되지 않습니다([2-5]).
- 보유율 5~95% 항목만, 동급 그룹은 보유·미보유가 각각 10건 이상인 것만 씁니다.

In [ ]:
# [4-1] 편의시설 → 거래 규칙
a = m[m["schema_new"]].copy()
a["sold"] = 1 - y.loc[a.index]
items_all = am.loc[a.index].explode()
support = items_all.value_counts() / len(a)
items = support[(support >= 0.05) & (support <= 0.95)].index
hit = items_all[items_all.isin(items)]
arr = np.zeros((len(a), len(items)), dtype=bool)
arr[a.index.get_indexer(hit.index), items.get_indexer(hit.values)] = True
H = pd.DataFrame(arr, index=a.index, columns=items)

base = a["sold"].mean()
peer = pd.read_parquet(ROOT / "data/processed/InsightStay_data_cleaned.parquet",
                       columns=["peer_group"]).loc[a.index, "peer_group"]
rows = []
for it in items:
    h = H[it]
    g = (pd.DataFrame({"g": peer, "h": h, "s": a["sold"]})
         .groupby(["g", "h"])["s"].agg(["mean", "size"]).unstack().dropna())
    g = g[(g[("size", True)] >= 10) & (g[("size", False)] >= 10)]
    w = g[("size", True)] + g[("size", False)]
    rows.append({"편의시설": it, "지지도": h.mean(), "신뢰도 P(거래|보유)": a["sold"][h].mean(),
                 "향상도": a["sold"][h].mean() / base,
                 "동급 내 거래율 차이": ((g[("mean", True)] - g[("mean", False)]) * w).sum() / w.sum(),
                 "L5 보유율": h[a["segment"] == "L5"].mean(), "L3 보유율": h[a["segment"] == "L3"].mean()})
rules = pd.DataFrame(rows).set_index("편의시설").sort_values("동급 내 거래율 차이", ascending=False)
print(f"새 이름 체계 매물 {len(a):,}건 · 거래율 {base:.1%} · 항목 {len(items)}개")
rules.head(15)

In [ ]:
# [4-2] 거래율과 반대로 가는 항목 — 휴양지형 설비(주차·에어컨)와 옛 이름이 섞여 있다
rules.tail(8)

**읽는 법**
- **동급 내 거래율 차이**가 핵심 지표입니다. 같은 도시·방 타입·인원·최소 숙박일 매물끼리 비교했을 때, 이 항목이 있는 매물의 거래율이 몇 %p 높은가입니다.
- 향상도가 1에 가까운 항목이 많은 것은 보유율이 높아서입니다. 이런 항목은 **"없으면 눈에 띄게 불리한 기본 구비품"** 으로 읽습니다.
- 연관이지 인과가 아닙니다. 알림 문구는 "이 항목을 들이면 팔린다"가 아니라 **"주변 거래 매물 대부분이 갖춘 항목이 없다"** 로 씁니다.

## 5. 기대 효과 — 바꾸면 얼마를 버나

```
기대 수수료 = Σ  전환확률 증가 × 호스트 반응률 × 전환 후 가동률 × 365 × 교정 후 1박 가격 × 15.5%
```

| 요소 | 어디서 오나 |
|---|---|
| 전환확률 증가 | [2] 로지스틱 모델로 "현재 특성"과 "동급 수준으로 개선한 특성"의 L3 확률 차이 |
| 동급 수준으로 개선 | 가격 `price_rel` → 1배 초과분만 1로 · 편의시설 `amenity_rel` → 부족분만 0으로 · 최소 숙박 5박 이상 → 3박 |
| 호스트 반응률 | **가정** 10 / 20 / 30% — 알림을 받고 실제로 고치는 호스트 비율 |
| 전환 후 가동률 | 같은 동급 그룹 **거래 매물의 리뷰 기반 가동률 중앙값** (보수적 — 회귀 예측값은 R²가 낮아 쓰지 않음, `[5-1]`) |
| 리뷰 작성률 | **가정** 0.3 / 0.5 / 0.7 — 가동률이 `0.5 ÷ 작성률` 배로 바뀜 |
| 1박 가격 | `price` 는 성수기 견적이라 결과는 **상한** |

**전환확률 증가는 모델이 말하는 연관의 크기이지 인과 효과가 아닙니다.** 그래서 반응률을 곱해 한 번 더 낮추고, 결과는 범위로만 제시합니다.

In [ ]:
# [5-1] 가동률 회귀 — 거래 매물(L4+L5)에서 occ_review 를 설명할 수 있나
from sklearn.linear_model import LinearRegression
occ = pd.read_parquet(ROOT / "data/processed/InsightStay_data_cleaned.parquet",
                      columns=["occ_review", "peer_group"]).loc[m.index]
sold = y == 0
reg = LinearRegression().fit(X[sold], occ.loc[sold, "occ_review"])
print(f"거래 매물 {sold.sum():,}건 · 가동률 회귀 R² = {reg.score(X[sold], occ.loc[sold, 'occ_review']):.3f}")
print("→ R²가 낮아 개별 매물의 가동률을 예측하기 어렵다. 전환 후 가동률은 동급 그룹 거래 매물의 중앙값으로 둔다.")
pd.Series(reg.coef_, index=X.columns, name="계수").round(3)

In [ ]:
# [5-2] 시나리오 계산
peer_occ = occ[sold].groupby("peer_group")["occ_review"].median()
price_all = pd.read_parquet(ROOT / "data/processed/InsightStay_data_cleaned.parquet",
                            columns=["price"]).loc[m.index, "price"]

def design(pr, ar, mnights, room, peak):
    mnb = pd.cut(mnights, [0, 1, 2, 3, 7, 29], labels=["1", "2", "3", "4-7", "8-29"])
    return pd.DataFrame({
        "동급 대비 가격 (log)": np.log(pr.clip(0.2, 5)), "동급 대비 편의시설 (+5개)": ar.clip(-40, 40) / 5,
        "최소 숙박 1박": (mnb == "1").astype(int), "최소 숙박 3박": (mnb == "3").astype(int),
        "최소 숙박 4-7박": (mnb == "4-7").astype(int), "최소 숙박 8-29박": (mnb == "8-29").astype(int),
        "개인실·호텔·공유": (room != "Entire home/apt").astype(int), "성수기(6-8월) 견적": peak.astype(int),
    }, index=pr.index)

def scenario(idx):
    t = m.loc[idx]
    pr1, ar1 = t["price_rel"].clip(upper=1), t["amenity_rel"].clip(lower=0)
    mn1 = t["minimum_nights"].where(t["minimum_nights"] < 5, 3)
    dp = (logit.predict_proba(design(t["price_rel"], t["amenity_rel"], t["minimum_nights"], t["room_type"], t["quote_peak"]))[:, 1]
          - logit.predict_proba(design(pr1, ar1, mn1, t["room_type"], t["quote_peak"]))[:, 1])
    occ1 = occ.loc[idx, "peer_group"].map(peer_occ).fillna(0)
    price1 = price_all.loc[idx] / t["price_rel"] * pr1
    rows = []
    for rr in [0.3, 0.5, 0.7]:
        fee = np.minimum(occ1 * 0.5 / rr, 0.7) * 365 * price1 * 0.155
        for resp in [0.1, 0.2, 0.3]:
            rows.append({"리뷰 작성률": rr, "반응률": resp, "전환(건)": (dp * resp).sum(),
                         "연 수수료 증가(M€)": (dp * resp * fee).sum() / 1e6})
    out = pd.DataFrame(rows)
    print(f"대상 {len(t):,}건 · 평균 전환확률 증가 {dp.mean():.3f} · "
          f"전환 1건당 연 수수료 중앙값 {np.median(np.minimum(occ1, 0.7) * 365 * price1 * 0.155):,.0f}€")
    return out.pivot(index="리뷰 작성률", columns="반응률", values="연 수수료 증가(M€)")

hot = leaves[leaves["L3비율"] >= 0.5].index
target_hot = leaf[leaf.isin(hot) & (y == 1)].index
print("① 트리 고위험 잎의 L3")
s_hot = scenario(target_hot)
s_hot

In [ ]:
print("② L3 전체 (신규 제외)")
s_all = scenario(y[y == 1].index)
s_all

In [ ]:
# [5-3] 규모 감각 — 같은 정의로 계산한 현재 플랫폼 수수료 대비
base_rows = pd.read_parquet(ROOT / "data/processed/InsightStay_data_cleaned.parquet",
                            columns=["source", "price", "occ_review"]).query("source == 0 and price == price")
current = (base_rows["occ_review"] * 365 * base_rows["price"] * 0.155).sum() / 1e6
print(f"현재 추정 수수료 (source=0, 성수기 가격 기준) {current:,.0f} M€")
print(f"① 기본 가정(작성률 0.5, 반응 20%) {s_hot.loc[0.5, 0.2]:.2f} M€ = {s_hot.loc[0.5, 0.2] / current:.2%}")
print(f"② 기본 가정(작성률 0.5, 반응 20%) {s_all.loc[0.5, 0.2]:.2f} M€ = {s_all.loc[0.5, 0.2] / current:.2%}")

In [ ]:
# [5-4] 방치된 기회의 크기 — 분모를 '플랫폼 전체 수수료'가 아니라 '되살릴 수 있었던 수익'으로
# 무거래·저활동 층이 동급 거래 매물의 가동률만 달성했을 때의 수수료. 전환 가능성이 아니라 기회의 상한이다.
full = pd.read_parquet(ROOT / "data/processed/InsightStay_data_cleaned.parquet",
                       columns=["source", "price", "occ_review", "peer_group", "segment",
                                "L3_new", "number_of_reviews", "number_of_reviews_ltm"])
s0 = (full["source"] == 0) & full["price"].notna()
traded_all = s0 & full["segment"].isin(["L4", "L5"])
fee_of = lambda d, o: (d["price"] * 365 * o * 0.155).sum()

occ_peer = full[traded_all].groupby("peer_group")["occ_review"].median()
current_fee = fee_of(full[traded_all], full.loc[traded_all, "occ_review"])

rows = []
for seg in ["L2", "L3", "L4"]:
    sel = s0 & (full["segment"] == seg)
    if seg == "L3":
        sel = sel & ~full["L3_new"]          # 신규 매물은 '안 팔린다'가 아니라 '판단 불가'
    d = full[sel].copy()
    d["occ_if"] = d["peer_group"].map(occ_peer)
    d = d[d["occ_if"].notna()]
    now, pot = fee_of(d, d["occ_review"]), fee_of(d, d["occ_if"])
    rows.append({"대상(건)": len(d), "현재(M€)": now / 1e6,
                 "동급 수준일 때(M€)": pot / 1e6, "방치된 기회(M€)": (pot - now) / 1e6})
gap = pd.DataFrame(rows, index=["L2 장기설정", "L3 단기무거래", "L4 저활동"])
gap.loc["합계"] = gap.sum()

print(f"현재 연 수수료 (거래 매물, source=0) {current_fee / 1e6:,.0f} M€")
print(f"방치된 기회 합계 = 현재 수수료의 {gap.loc['합계', '방치된 기회(M€)'] * 1e6 / current_fee:.0%}")
gap.round(1)

In [ ]:
# [5-5] 전환이 1년만 가는가 · 알림 1건당 회수액
# 한 번 팔리기 시작한 매물이 그 해만 팔리고 마는지 확인한다. 누적 리뷰가 최근 1년의 몇 배인지로 본다.
l5 = full[s0 & (full["segment"] == "L5")]
cum, ltm_med = l5["number_of_reviews"].median(), l5["number_of_reviews_ltm"].median()
print(f"L5 누적 리뷰 중앙값 {cum:.0f}건 / 최근 1년 {ltm_med:.0f}건  →  약 {cum / ltm_med:.1f}년치 축적")
print(f"L5 중 누적이 최근 1년의 3배 이상인 비율 {(l5['number_of_reviews'] >= l5['number_of_reviews_ltm'] * 3).mean():.1%}")
print("→ 거래가 한 해만 일어나는 것이 아니므로, 회수액을 1년으로만 세면 과소 추정이다.\n")

# 알림 1건당 회수액 — 비용이 거의 0인 개입이라 ROI 관점에서 제시한다
for label, s, n in [("① 고위험 L3", s_hot, len(target_hot)), ("② L3 전체", s_all, int((y == 1).sum()))]:
    g = s.loc[0.5, 0.2]          # 기본 가정: 리뷰 작성률 0.5 · 반응률 20%
    print(f"{label}  대상 {n:>7,}건 · 연 {g:5.2f} M€ · 알림 1건당 {g * 1e6 / n:>5,.0f}€ · 3년 누적 {g * 3:5.1f} M€")

**읽는 법** — 절대 금액은 가정에 따라 몇 배씩 움직이므로 **범위**로 봅니다. 그리고 회수액을 **"플랫폼 전체 수수료의 몇 %"로 말하지 않습니다.** 분모가 1,179 M€라 어떤 개입도 1% 미만으로 보이는데, 문제가 작은 것이 아니라 분모를 잘못 고른 것입니다.

발표에서 짚을 것:
- **문제의 크기와 회수액을 분리**합니다. 방치된 기회는 `[5-4]`, 회수액과 1건당 효율은 `[5-5]`입니다.
- 개입이 리스팅 갱신 알림이라 **비용이 거의 0**입니다. 비율보다 **알림 1건당 회수액**이 의사결정에 맞는 지표입니다.
- L3는 노출·지원 자원을 쓰면서 수수료가 0인 재고입니다(과제 비즈니스 규칙). 반응하지 않는 매물의 노출 순위 조정과 묶으면 비용 측면 논리가 섭니다. 이 비용은 계량할 수 없어 정성 근거로 둡니다.
- **가정을 유리하게 바꾸지 않습니다.** 위 표에서 반응률 30% · 작성률 0.3을 고르면 12.6 M€가 나오지만, 근거가 없어 쓰지 않습니다. 기본 가정(반응률 20% · 작성률 0.5)에서도 성립한다는 것이 논지입니다.

## 결과 요약

실행 후 아래 수치를 [`reports/MODELING_PLAN.md`](../reports/MODELING_PLAN.md) 1~5단계 결과 칸에 옮깁니다.

| 확인할 것 | 셀 |
|---|---|
| 층 차이의 효과 크기(ε², Cramér V) | `[1-2]` |
| 편의시설 vs 가격 표준화 계수 (이름 체계 통제 후 포함) | `[2-2]` `[2-5]` |
| 무작위 / 도시 단위 AUC | `[2-3]` |
| 트리 분기점과 안정성 | `[3-1]` `[3-3]` |
| 최악 조합 잎의 L3 비율 | `[3-2]` |
| 알림 문구에 넣을 편의시설 | `[4-1]` |
| 기대 수수료 범위와 현재 대비 비율 | `[5-2]` `[5-3]` |